# AI Gym Coach: Kinematic Angle & Trajectory Analysis

This notebook analyzes the joint kinematics (knee flexion, hip hinge, torso lean) extracted from exercise video clips to establish data-driven thresholds for the squat repetition state machine.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
data_path = os.path.join("..", "data", "processed", "squat_dataset.csv")
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} frames across classes: {df['label'].unique().tolist()}")
df.head()

## 1. Class Distribution & Kinematic Summary

In [ ]:
summary = df.groupby("label")[["knee_angle", "back_lean", "hip_angle"]].agg(["count", "mean", "std", "min", "max"])
summary

## 2. Knee Angle and Back Lean Trajectories Across Frames

In [ ]:
good_df = df[df["clip"] == "squat_sample_good"].sort_values("frame")
shallow_df = df[df["clip"] == "squat_shallow_rep"].sort_values("frame")

plt.figure(figsize=(10, 5))
plt.plot(good_df["frame"], good_df["knee_angle"], label="Good Squat (Full Depth)", color="#00bcd4", lw=2.5)
plt.plot(shallow_df["frame"], shallow_df["knee_angle"], label="Shallow Squat (Half Rep)", color="#ff9800", lw=2, linestyle="--")
plt.axhline(160, color="#4caf50", linestyle=":", label="Standing Threshold (160 deg)")
plt.axhline(95, color="#e91e63", linestyle=":", label="Full Depth Threshold (<=95 deg)")
plt.title("Knee Flexion Trajectory: Good vs Shallow Squat")
plt.xlabel("Frame Index")
plt.ylabel("Knee Angle (degrees)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()

## 3. Threshold Justification Matrix for Viva

| State | Metric | Threshold | Biomechanical Rationale |
|---|---|---|---|
| **Standing** | Knee Angle | $\ge 160^\circ$ | Upright human standing posture |
| **Descending** | Knee Angle | Drops below $150^\circ$ | Hysteresis prevents false starts from small body shifts |
| **Good Depth** | Knee Angle | $\le 95^\circ$ | Thigh breaks parallel with floor |
| **Too Shallow** | Knee Angle | Turns up $> 95^\circ$ | Incomplete range of motion (half squat) |
| **Excessive Lean** | Back Lean | $> 40^\circ$ | Risk of excessive spinal load / bad posture |